# DIMER Workshop: Comparing Modern Image Classification Architectures

**Profile:** `E2E` · **Mode:** `WORKSHOP` · **Notebook Specification:** 2.1 · **Standalone:** yes

This workshop compares six **live DIMER image-classification models** under one transfer-learning protocol:

- ResNet-50
- MobileNetV4-Conv-Small
- ConvNeXt-Tiny
- ViT-B/16
- SwinV2-Tiny
- EVA-02 Base 448

The comparison question is deliberately narrow:

> **When every pretrained backbone is frozen and receives the same labelled images, how useful are its learned representations for a new six-class bird-classification task?**

Each model gets the same train/validation/test split, the same linear-head training budget, the same model-selection rule, and the same metrics. The notebook exports one small SafeTensors classification-head adapter per backbone and verifies each adapter by fresh reload.

This is a comparative workshop, not a replacement for the model-specific DIMER tutorials. Those primary notebooks may use different fine-tuning policies.


## Learning goals

You will:

1. compare a residual CNN, an efficiency-oriented ConvNet, a modern ConvNet, a plain Vision Transformer, a hierarchical transformer, and EVA-02 under one protocol;
2. separate **pretrained representation quality** from architecture-specific fine-tuning policy;
3. use validation macro-F1 for model selection and keep the final test split independent;
4. compare predictive quality with parameter count, feature-extraction time, and input resolution;
5. export and fresh-reload the actual linear-head adapters created by the workshop; and
6. understand why a single small transfer-learning exercise is tutorial evidence rather than a general architecture ranking.


In [ ]:
# @title 0. Workshop controls and runtime checks
USE_BYOD = False  # @param {type:"boolean"}
BYOD_ZIP_PATH = ""  # @param {type:"string"}
BATCH_SIZE = 16  # @param {type:"integer"}
HEAD_EPOCHS = 150  # @param {type:"integer"}
HEAD_LR = 0.01  # @param {type:"number"}
PLOT_MODEL = "vit"  # @param ["resnet50","mobilenetv4","convnext","vit","swinv2","eva02"]

import csv
import hashlib
import importlib.metadata
import io
import json
import os
import platform
import random
import stat
import subprocess
import sys
import time
import urllib.request
import venv
import zipfile
from pathlib import Path, PurePosixPath

from packaging.version import Version

if sys.version_info[:2] != (3, 12):
    raise RuntimeError(f"This reviewed workshop supports Python 3.12; found {platform.python_version()}.")

def require_version(name, minimum, maximum):
    value = Version(importlib.metadata.version(name))
    if not (Version(minimum) <= value < Version(maximum)):
        raise RuntimeError(f"{name} {value} is outside the reviewed range [{minimum}, {maximum}).")
    return str(value)

CONTROL_VERSIONS = {
    "numpy": require_version("numpy", "1.26", "3.0"),
    "pandas": require_version("pandas", "2.2", "4.0"),
    "matplotlib": require_version("matplotlib", "3.8", "4.0"),
    "packaging": require_version("packaging", "24.0", "27.0"),
}

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

if BATCH_SIZE < 1 or BATCH_SIZE > 64:
    raise ValueError("BATCH_SIZE must be in 1..64.")
if HEAD_EPOCHS < 10 or HEAD_EPOCHS > 1000:
    raise ValueError("HEAD_EPOCHS must be in 10..1000.")
if not 0 < HEAD_LR <= 0.1:
    raise ValueError("HEAD_LR must be in (0, 0.1].")

def gpu_info():
    try:
        return subprocess.check_output(
            ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"],
            text=True, stderr=subprocess.STDOUT
        ).strip()
    except Exception:
        return ""

GPU_INFO = gpu_info()
if not GPU_INFO:
    raise RuntimeError(
        "The default six-model workshop is GPU-oriented. Select a CUDA GPU runtime before Run all."
    )

ROOT = Path.cwd()
RUNTIME_ROOT = ROOT / "workshop_runtime"
DATA_ROOT = RUNTIME_ROOT / "image_classification_data"
OUTPUT_ROOT = ROOT / "outputs"
ARTIFACT_ROOT = OUTPUT_ROOT / "classification_adapters"
for path in (RUNTIME_ROOT, DATA_ROOT, OUTPUT_ROOT, ARTIFACT_ROOT):
    path.mkdir(parents=True, exist_ok=True)

print({"python": platform.python_version(), "control_versions": CONTROL_VERSIONS, "gpu": GPU_INFO})


## 1. Exact live-model identities

The workshop loads the exact upstream checkpoints represented by the live DIMER profiles and pins each one to its immutable revision.

All six checkpoints are loaded through `timm` from Hugging Face at the named commit. The notebook verifies the Hub's resolved commit before loading. Remote model-repository code is not executed.

The workshop uses one reviewed runtime stack for the comparative experiment: PyTorch 2.14 and timm 1.0.29. Five live carriers already use that stack; the SwinV2 carrier pins timm 1.0.28, so this workshop explicitly records that the **weights/revision are identical but the comparison runtime uses timm 1.0.29**.


In [ ]:
# @title 1.1 Model registry
MODELS = {
    "resnet50": {
        "display_name": "ResNet-50",
        "model_id": "timm/resnet50.a1_in1k",
        "revision": "767268603ca0cb0bfe326fa87277f19c419566ef",
        "license": "Apache-2.0",
        "family": "Residual CNN",
    },
    "mobilenetv4": {
        "display_name": "MobileNetV4-Conv-Small",
        "model_id": "timm/mobilenetv4_conv_small.e2400_r224_in1k",
        "revision": "331fb803779522b685cf942e15f914fb6741c1eb",
        "license": "Apache-2.0",
        "family": "Efficiency-oriented ConvNet",
    },
    "convnext": {
        "display_name": "ConvNeXt-Tiny",
        "model_id": "timm/convnext_tiny.in12k_ft_in1k",
        "revision": "aa096f03029c7f0ec052013f64c819b34f8ad790",
        "license": "Apache-2.0",
        "family": "Modern ConvNet",
    },
    "vit": {
        "display_name": "ViT-B/16",
        "model_id": "timm/vit_base_patch16_224.orig_in21k_ft_in1k",
        "revision": "e0bd370de6799e8d1f47a911174ff4c3708e2323",
        "license": "Apache-2.0",
        "family": "Vision Transformer",
    },
    "swinv2": {
        "display_name": "SwinV2-Tiny",
        "model_id": "timm/swinv2_tiny_window8_256.ms_in1k",
        "revision": "650d02aabf05e8adbd060a739ab39e39f53da639",
        "license": "MIT",
        "family": "Hierarchical/windowed transformer",
        "runtime_note": "live carrier pins timm 1.0.28; workshop uses timm 1.0.29",
    },
    "eva02": {
        "display_name": "EVA-02 Base 448",
        "model_id": "timm/eva02_base_patch14_448.mim_in22k_ft_in22k_in1k",
        "revision": "81063ecfe9c381a16a19d06f396d6c7011aa426a",
        "license": "MIT",
        "family": "EVA-02 Vision Transformer",
    },
}
display(pd.DataFrame([{"key": key, **value} for key, value in MODELS.items()]))


## 2. Dataset provenance and BYOD contract

The default dataset is the same real, pinned sample already used by the live ViT transfer-learning carrier:

- **180 research-grade iNaturalist photographs**
- six common North American bird species
- 30 photographs per species
- one photograph per observer per species
- each served image pinned by byte count and SHA-256
- **CC0 1.0** photographs
- deterministic split seed 42: **108 train / 24 validation / 48 test**

The photographs are fetched from the iNaturalist open-data bucket and are not redistributed by this notebook.

The test split remains untouched until the validation-based model choice has been frozen.

### BYOD

Set `USE_BYOD=True` and provide a local ZIP path in `BYOD_ZIP_PATH`. The ZIP must contain `labels.csv` with:

`id,file,label,split`

where `split` is exactly `train`, `validation`, or `test`. Every file path must be relative and point to one image inside the ZIP. The notebook preserves these pre-assigned splits rather than silently re-splitting them.

The BYOD archive is extracted locally with path-traversal, absolute-path, symlink, expanded-size, and duplicate-file checks. User images do not leave the notebook runtime.


In [ ]:
# @title 2.1 Pinned iNaturalist sample manifest
SAMPLE_SEED = 42
SAMPLE_SPLIT = {"train": 18, "validation": 4, "test": 8}  # per class
CORPUS_BASE_URL = "https://inaturalist-open-data.s3.amazonaws.com/photos/"
CORPUS_LICENSE = "CC0 1.0"
SAMPLE_RECORDS = [{"id":"song_sparrow-00","label":"song_sparrow","photo_id":129376982,"observation_id":79016324,"observer":"andywilson","bytes":43427,"sha256":"7a9d9304a82f202e992655ec5f65477cd3d7c1dce03aa89a214c2daa38f9d61d","ext":"jpg"},{"id":"song_sparrow-01","label":"song_sparrow","photo_id":480991086,"observation_id":267636534,"observer":"lyneisfilm","bytes":162073,"sha256":"11f77ff277dd2703c1000f2c787136ff0c3ca7ffad7017056892fe789d65efec","ext":"jpg"},{"id":"song_sparrow-02","label":"song_sparrow","photo_id":546060381,"observation_id":302980489,"observer":"swpollinators","bytes":27899,"sha256":"4e70b9519c6e5f7384a4495b91b45465f2b1599f86491d8f9f9b12635a4046f6","ext":"jpg"},{"id":"song_sparrow-03","label":"song_sparrow","photo_id":308625896,"observation_id":177450028,"observer":"radrat","bytes":70961,"sha256":"1211da4fdb24ae85ef0c6c3e2d03542c430457856aec661fe8f5f2de0027eee5","ext":"jpeg"},{"id":"song_sparrow-04","label":"song_sparrow","photo_id":494793016,"observation_id":275349085,"observer":"k-simpkins","bytes":58410,"sha256":"255538cf450197257e86ed3d41dc69fb78e594434e9cb338c6314288c6cff26e","ext":"jpg"},{"id":"song_sparrow-05","label":"song_sparrow","photo_id":674054489,"observation_id":369029444,"observer":"ben142","bytes":220573,"sha256":"1ae24622888d9d449ffd6b5c65cac1b9b14870fd8e12dbed8aa0acc2f5030123","ext":"jpg"},{"id":"song_sparrow-06","label":"song_sparrow","photo_id":339623726,"observation_id":193339933,"observer":"rawcomposition","bytes":25012,"sha256":"d2cde085277a71886a2bf211a1eec26941a73752375708726a8af29aa4995a8b","ext":"jpg"},{"id":"song_sparrow-07","label":"song_sparrow","photo_id":222768957,"observation_id":130949329,"observer":"davidfbird","bytes":110773,"sha256":"0feee62753f409d0aa365e9aa017ddef436ad70a2673dc847feb3b5386af27ff","ext":"jpg"},{"id":"song_sparrow-08","label":"song_sparrow","photo_id":181658744,"observation_id":107953669,"observer":"gcart043","bytes":98482,"sha256":"a662a6abb24f42b256fb6e2d6f02c3e128a1053ef534f454461aa5cadcc03fe4","ext":"jpeg"},{"id":"song_sparrow-09","label":"song_sparrow","photo_id":148994242,"observation_id":90171417,"observer":"glennberry","bytes":102702,"sha256":"64333977d24957d723a1d26886004f222d810557617685579f72a6b553e508fa","ext":"jpg"},{"id":"song_sparrow-10","label":"song_sparrow","photo_id":637315932,"observation_id":349374463,"observer":"sooji","bytes":136572,"sha256":"a5cebbc0cc2325d3805c4ac854e103f7ba1c22e3a34fb204f30d58681e865033","ext":"jpg"},{"id":"song_sparrow-11","label":"song_sparrow","photo_id":471146686,"observation_id":262252507,"observer":"jeanpaulboerekamps","bytes":98601,"sha256":"4301f06b52b8dcf1e137567c32412e384edb71cb90e2e35459a0405b2e56529b","ext":"jpg"},{"id":"song_sparrow-12","label":"song_sparrow","photo_id":123859010,"observation_id":75689904,"observer":"w_mark_c","bytes":190819,"sha256":"6b6057a1c50b83ffeb4d9e34367b3e6a9b355236dbcae9820b509e1484f8fe33","ext":"jpg"},{"id":"song_sparrow-13","label":"song_sparrow","photo_id":640811426,"observation_id":351179648,"observer":"erikschiff","bytes":107695,"sha256":"27aecce184a485ee888c0e8101cb4a34899b8a185b62dc064f3dc2ec9902182b","ext":"jpg"},{"id":"song_sparrow-14","label":"song_sparrow","photo_id":63537800,"observation_id":40010230,"observer":"nathanael15","bytes":51441,"sha256":"5ae55f868e779a4e8ee34f6ad077b40c41f20aa699d967f373ebd659a89137a8","ext":"jpg"},{"id":"song_sparrow-15","label":"song_sparrow","photo_id":108520869,"observation_id":67204020,"observer":"dugald","bytes":52496,"sha256":"e483364889fb95c540db84b5edf5a2400febf7d2eb62a1e49303b13ad329d1b8","ext":"jpg"},{"id":"song_sparrow-16","label":"song_sparrow","photo_id":435251292,"observation_id":244042351,"observer":"carterdorscht","bytes":166662,"sha256":"d0cdd9ddcf7202a91ac2c47910a0c230639bce50283e8511fb8311151763233a","ext":"jpeg"},{"id":"song_sparrow-17","label":"song_sparrow","photo_id":120710033,"observation_id":73898230,"observer":"tys_rbg","bytes":126820,"sha256":"515b3b32b64e88d401990bfd1d8e2c1281443b5d1e763a09e86d3d3664e1df41","ext":"jpg"},{"id":"song_sparrow-18","label":"song_sparrow","photo_id":393408927,"observation_id":222067370,"observer":"irenemacaulay_","bytes":101681,"sha256":"22e326807de963352b4307790bd40c5506adb0fab1f9844968edaf4bc5665e1c","ext":"jpg"},{"id":"song_sparrow-19","label":"song_sparrow","photo_id":349361283,"observation_id":198243065,"observer":"sean579","bytes":42820,"sha256":"5eb0031a8d6066650c66b265fb1724413273e095e4e531054d2817eb75910074","ext":"jpg"},{"id":"song_sparrow-20","label":"song_sparrow","photo_id":608802621,"observation_id":335154467,"observer":"jamesadney","bytes":112564,"sha256":"6e94bce1b5135f48b0b19b64e21a0c76cff4f9b9a28fcbc2b7f5c8ddda6ef513","ext":"jpg"},{"id":"song_sparrow-21","label":"song_sparrow","photo_id":614258628,"observation_id":337847585,"observer":"joy4birds","bytes":70552,"sha256":"aa767aa5a74a76cfd985aba5282e589f670a9ce0a8fcdc6a096c0357992dec67","ext":"jpg"},{"id":"song_sparrow-22","label":"song_sparrow","photo_id":634100352,"observation_id":347744524,"observer":"zorthesosen","bytes":214044,"sha256":"6a70dba26dfb3e98a244a9ec9badb812ddaa8b3612e7a36f66aba057bb757ecd","ext":"jpg"},{"id":"song_sparrow-23","label":"song_sparrow","photo_id":50065474,"observation_id":31954532,"observer":"truthseqr","bytes":82521,"sha256":"4186fbf344e92038358d4338102aa440098bff5f558ab1d197f72fefbcec0b15","ext":"jpeg"},{"id":"song_sparrow-24","label":"song_sparrow","photo_id":8656044,"observation_id":6803564,"observer":"glmory","bytes":297964,"sha256":"e8cab773436ccfaad4699e5112ef4edb516236d413d6dbf034eea7bf9c88c8f5","ext":"jpg"},{"id":"song_sparrow-25","label":"song_sparrow","photo_id":131051149,"observation_id":79988590,"observer":"funvill","bytes":72572,"sha256":"0a694b5bb03aee6eb5a3a6132e855d47172b21b84cc4cc7c64367d1a31de7894","ext":"jpeg"},{"id":"song_sparrow-26","label":"song_sparrow","photo_id":12923156,"observation_id":9491600,"observer":"gambolingquail","bytes":69559,"sha256":"6a92bff4fc76820c6f21e15c5f254385dd976a32264911bfc08417a1c2bf053d","ext":"jpg"},{"id":"song_sparrow-27","label":"song_sparrow","photo_id":12077500,"observation_id":8959545,"observer":"reuvenm","bytes":74010,"sha256":"3e3c5e94c839f45610ef3ef7ffaf3575f4bfe365fc94454c30bfdb718ca3c593","ext":"jpeg"},{"id":"song_sparrow-28","label":"song_sparrow","photo_id":132730299,"observation_id":80955309,"observer":"steph123456","bytes":154309,"sha256":"5f8854dd231a302c643b22521b49ed3007203d74b8b8413a23abc79ffe0b0270","ext":"jpeg"},{"id":"song_sparrow-29","label":"song_sparrow","photo_id":124840110,"observation_id":76316566,"observer":"terrimewbornagain","bytes":81046,"sha256":"2020092b0b67397a78cc2b0df267fb671b3ba18e03fb0e35e664e0784061e3c6","ext":"jpeg"},{"id":"chipping_sparrow-00","label":"chipping_sparrow","photo_id":198992636,"observation_id":117809422,"observer":"k-simpkins","bytes":62563,"sha256":"cf9f3b0c1863808e21af596b2e609b047ddbc28cb2ed076625e2546425ff0adf","ext":"jpg"},{"id":"chipping_sparrow-01","label":"chipping_sparrow","photo_id":248210057,"observation_id":144599194,"observer":"w_mark_c","bytes":193389,"sha256":"497d0a0fef81c326bcc87b5d1eb97fe987d559b8f2b71bb60fe422ae34dce150","ext":"jpg"},{"id":"chipping_sparrow-02","label":"chipping_sparrow","photo_id":156350853,"observation_id":94266719,"observer":"ellyne","bytes":142332,"sha256":"6a60ebac34476a372b4790a87d823432cdda8f72590930b5d18ae166cf4c7ba2","ext":"jpeg"},{"id":"chipping_sparrow-03","label":"chipping_sparrow","photo_id":16128796,"observation_id":11327134,"observer":"reuvenm","bytes":70532,"sha256":"5ad36c9cdd6c92e225a1b8ab3c04d2f65bc4e971d0243958f8ac090b0996115c","ext":"jpeg"},{"id":"chipping_sparrow-04","label":"chipping_sparrow","photo_id":391300648,"observation_id":220982684,"observer":"carterdorscht","bytes":208567,"sha256":"c974a676c3c227c2844ff822d429c784bc87bb41f40d5074a0ebadd4c6785b21","ext":"jpeg"},{"id":"chipping_sparrow-05","label":"chipping_sparrow","photo_id":339456083,"observation_id":193252042,"observer":"rawcomposition","bytes":46329,"sha256":"41c9260ca9107430e3a8090ba01cebf3e3c25f2dad15bea4f77c8e824d9fc496","ext":"jpg"},{"id":"chipping_sparrow-06","label":"chipping_sparrow","photo_id":40457652,"observation_id":26076708,"observer":"andywilson","bytes":156894,"sha256":"b70edd35e00fe672a39d5f0441ea4e9bb9fac19b0f2415ae13e22160bc6091a6","ext":"jpeg"},{"id":"chipping_sparrow-07","label":"chipping_sparrow","photo_id":84151914,"observation_id":52921135,"observer":"davidfbird","bytes":145714,"sha256":"ea65ce5ed881ded9a8157b5756fa907948f41e0eefb6735b48a1c43993e977f9","ext":"jpeg"},{"id":"chipping_sparrow-08","label":"chipping_sparrow","photo_id":523674610,"observation_id":291074747,"observer":"rwp84","bytes":47983,"sha256":"41eac0a5fb578b089f7524c4d2e6cb38c5508aa81815d6cfc46948c081daa800","ext":"jpg"},{"id":"chipping_sparrow-09","label":"chipping_sparrow","photo_id":292695018,"observation_id":168861389,"observer":"tim_kirsten","bytes":64812,"sha256":"cbb2a03f5dbd2209d56f1cca8b49f342dfbaf44e51fe1014ead75b2018c6678d","ext":"jpeg"},{"id":"chipping_sparrow-10","label":"chipping_sparrow","photo_id":478480946,"observation_id":266314208,"observer":"russnamitz","bytes":61359,"sha256":"8200cc2ec779d47be1b5afa261d934910a251a39f5620d3dc705106fb2bf5e0b","ext":"jpeg"},{"id":"chipping_sparrow-11","label":"chipping_sparrow","photo_id":220257388,"observation_id":129611350,"observer":"gcart043","bytes":129377,"sha256":"1bdce32e7ac58321dd6beacc084afaf45973469215d66b61a0f7c612da3db361","ext":"jpeg"},{"id":"chipping_sparrow-12","label":"chipping_sparrow","photo_id":92501489,"observation_id":57964052,"observer":"tniernberger","bytes":78380,"sha256":"44ddb9923026a97ee77ed362bc944c3d2cbbe4fbd4636000880e81613634e6c5","ext":"jpg"},{"id":"chipping_sparrow-13","label":"chipping_sparrow","photo_id":300376697,"observation_id":172991803,"observer":"matthias55","bytes":81986,"sha256":"7a9a5cbfb6a7d0581c98c75b5fdeb53a049d60f352eda43a39b1f9c2a3347f3b","ext":"jpeg"},{"id":"chipping_sparrow-14","label":"chipping_sparrow","photo_id":58192408,"observation_id":36778771,"observer":"bradenjudson","bytes":22352,"sha256":"7c630d7a5b24d94e677e563a8ecb36f57fb29babe6ac1568f11537f10a5edf75","ext":"jpeg"},{"id":"chipping_sparrow-15","label":"chipping_sparrow","photo_id":80410971,"observation_id":50636049,"observer":"radrat","bytes":55646,"sha256":"0c71ca735502e8c94db81302ecd006428206eb16d75472eb0c706e62e2656667","ext":"jpg"},{"id":"chipping_sparrow-16","label":"chipping_sparrow","photo_id":538480223,"observation_id":298914072,"observer":"hiltonward","bytes":201271,"sha256":"2a6479556f14a20a8c9a69ff0c1026fe4deb376944234d1d4429c558042c4346","ext":"jpg"},{"id":"chipping_sparrow-17","label":"chipping_sparrow","photo_id":370917657,"observation_id":209626382,"observer":"craigmartin","bytes":101736,"sha256":"feb94d319fe5b12c01e63a80dc8e44c45e15a0bfb6534ac5d2c0641205dfe1d5","ext":"jpg"},{"id":"chipping_sparrow-18","label":"chipping_sparrow","photo_id":264119989,"observation_id":152960401,"observer":"laurelthrone","bytes":194107,"sha256":"13e7336628f9ad784757dca82557c79eda22fdce561e0b52a10a503e72979b8e","ext":"jpeg"},{"id":"chipping_sparrow-19","label":"chipping_sparrow","photo_id":220323755,"observation_id":129631264,"observer":"enspring","bytes":85687,"sha256":"886b7e6faa39943f0c9754e4aef637e2a2f5ad57fe8e8c7ec08360ea95af45b1","ext":"jpeg"},{"id":"chipping_sparrow-20","label":"chipping_sparrow","photo_id":210319996,"observation_id":124105091,"observer":"bunnymom20","bytes":188328,"sha256":"2f6b6f7ac5ed9d91a361c8fed102484f4fca599ec39c9a2bfe9af1df53b28899","ext":"jpeg"},{"id":"chipping_sparrow-21","label":"chipping_sparrow","photo_id":99819947,"observation_id":62311494,"observer":"andy71","bytes":350329,"sha256":"d649c3c9fd6b0b159a979572b48daba39fc5608104f21c6d88c7d10fa2479f7e","ext":"jpg"},{"id":"chipping_sparrow-22","label":"chipping_sparrow","photo_id":480169680,"observation_id":267207764,"observer":"cvharris","bytes":144339,"sha256":"d1ab8643c48f2ea823f9b61843016795def21b290c35ce3d6045e4933b5dfa0a","ext":"jpeg"},{"id":"chipping_sparrow-23","label":"chipping_sparrow","photo_id":323624945,"observation_id":185313750,"observer":"mrspteranodon","bytes":232126,"sha256":"dd60e466086350ce1b9f7a9ba7784fc3963b3f996326ccf52f5adffa5719c39d","ext":"jpeg"},{"id":"chipping_sparrow-24","label":"chipping_sparrow","photo_id":343116928,"observation_id":195090930,"observer":"umamimomma","bytes":52983,"sha256":"a8948a0189d19b3d7b8df65271f4844b14bd5118f510d0b9ef458c942ac41ce2","ext":"jpg"},{"id":"chipping_sparrow-25","label":"chipping_sparrow","photo_id":354617430,"observation_id":200940828,"observer":"wafflemaster135","bytes":58919,"sha256":"2ba9557d06a7f1bcb1c20908efc82ea6317e5a0bd0c858898f3b5c0a007d20fb","ext":"jpeg"},{"id":"chipping_sparrow-26","label":"chipping_sparrow","photo_id":352953833,"observation_id":200088501,"observer":"aster-asti","bytes":105786,"sha256":"82c35e04e4e22fe35c9b364bce7737fa5226cb5c469c761adfc61ef0956da9db","ext":"jpg"},{"id":"chipping_sparrow-27","label":"chipping_sparrow","photo_id":465554743,"observation_id":259357260,"observer":"perrydise_koisplash","bytes":171532,"sha256":"35c6469fad62f198c060e056bd298f87976b39055cf66601a265ab55d5562642","ext":"jpeg"},{"id":"chipping_sparrow-28","label":"chipping_sparrow","photo_id":148831027,"observation_id":90084486,"observer":"ian-wolfe","bytes":181942,"sha256":"cfab51f0c0598120f5312b354ae249bafb15124dd74723a8c29cdd7319206e49","ext":"jpg"},{"id":"chipping_sparrow-29","label":"chipping_sparrow","photo_id":192945976,"observation_id":114256514,"observer":"sooji","bytes":200440,"sha256":"04fd7d05d5ced3073d4c7ce8a4f659c6994bb485c5253b26fbc471452b4a0b14","ext":"jpeg"},{"id":"white_throated_sparrow-00","label":"white_throated_sparrow","photo_id":339621218,"observation_id":193338380,"observer":"rawcomposition","bytes":31152,"sha256":"d1c08bfaca721bf0873437455b4cc010c6860d08b4777136c775007b0b9d07b6","ext":"jpg"},{"id":"white_throated_sparrow-01","label":"white_throated_sparrow","photo_id":166821399,"observation_id":99992799,"observer":"dziakj1","bytes":125954,"sha256":"c595a41fbc8948b0d918b59117340dc320e2ba80d29e92bb9dacaaed5b404852","ext":"jpeg"},{"id":"white_throated_sparrow-02","label":"white_throated_sparrow","photo_id":469820434,"observation_id":261505977,"observer":"joy4birds","bytes":111767,"sha256":"7ce091492c73c68395667bb45578dfff11457511b0958d1d0d320d1df3e55ceb","ext":"jpg"},{"id":"white_throated_sparrow-03","label":"white_throated_sparrow","photo_id":99351488,"observation_id":62040646,"observer":"bradenjudson","bytes":23399,"sha256":"e103968e2a6c9efb6f0bcb548a6457aef950a4a720838a624b6796b90411538e","ext":"jpeg"},{"id":"white_throated_sparrow-04","label":"white_throated_sparrow","photo_id":177497964,"observation_id":105746665,"observer":"andywilson","bytes":29827,"sha256":"65475d4842396f2488167453192d4aa834d2f1b40bcc78b420878c55ccf694d7","ext":"jpeg"},{"id":"white_throated_sparrow-05","label":"white_throated_sparrow","photo_id":628148203,"observation_id":344731686,"observer":"lavenderdame","bytes":106872,"sha256":"36379abf3af51d865ba6e6804ba3dd48fc9efd12ecc0b7d97e03fc04a16178a8","ext":"jpg"},{"id":"white_throated_sparrow-06","label":"white_throated_sparrow","photo_id":104660609,"observation_id":65043951,"observer":"allan7","bytes":42443,"sha256":"10791891945e83a0c908c14fea07a257a0f25f51c0e708bee35184213833a635","ext":"jpeg"},{"id":"white_throated_sparrow-07","label":"white_throated_sparrow","photo_id":250718938,"observation_id":145903421,"observer":"stevestevens","bytes":138735,"sha256":"bd52e6d2f48c247f72db0fa393ec4fa13af8510c95f0ca9134cbef71caddb6d4","ext":"jpeg"},{"id":"white_throated_sparrow-08","label":"white_throated_sparrow","photo_id":15105971,"observation_id":10793852,"observer":"schylerbrown","bytes":31467,"sha256":"6496e7e6d3abf13b1a538f769e3cc402280cf1e9a2a1ebdf81c68dcfd7ed01e2","ext":"jpeg"},{"id":"white_throated_sparrow-09","label":"white_throated_sparrow","photo_id":171460784,"observation_id":102554447,"observer":"w_mark_c","bytes":251287,"sha256":"3828c41af9209b408fe0d8ec6541edf35d13fc730b74fd27a82980d4f3771137","ext":"jpg"},{"id":"white_throated_sparrow-10","label":"white_throated_sparrow","photo_id":244260317,"observation_id":142471526,"observer":"deejay","bytes":75694,"sha256":"d67efb1ec61f6700b8a6c6552e2da9cd981e68ae81af16d6f1e0ef17fcaff84f","ext":"jpeg"},{"id":"white_throated_sparrow-11","label":"white_throated_sparrow","photo_id":194732675,"observation_id":115373159,"observer":"wildreturn","bytes":128989,"sha256":"7ed4bde480b35734576bb4c5f9d1453e77c1f095380432b43b1b682050255831","ext":"jpeg"},{"id":"white_throated_sparrow-12","label":"white_throated_sparrow","photo_id":341990462,"observation_id":194541980,"observer":"ethologist","bytes":149260,"sha256":"3e2710fac082cc347e6cd114d42d39f25ec47b82c25936922232eba916808cdb","ext":"jpg"},{"id":"white_throated_sparrow-13","label":"white_throated_sparrow","photo_id":267625208,"observation_id":154862375,"observer":"laurelthrone","bytes":148616,"sha256":"0cac1bc7053c891ce5ae1c33e3f94b69f1b19b958bd08c686db2ccaa7ba0fbd1","ext":"jpeg"},{"id":"white_throated_sparrow-14","label":"white_throated_sparrow","photo_id":330539586,"observation_id":188793537,"observer":"efalquet","bytes":119214,"sha256":"c3db4583124472b28dbfb4c6fd9b3829e451d508a6b0a095eccc28f17e7982b4","ext":"jpeg"},{"id":"white_throated_sparrow-15","label":"white_throated_sparrow","photo_id":193091403,"observation_id":114346779,"observer":"ian-wolfe","bytes":73751,"sha256":"e28974c8782fccb00f5ea5420140e01248ead859f151563a95a26bf67acba479","ext":"jpg"},{"id":"white_throated_sparrow-16","label":"white_throated_sparrow","photo_id":691050773,"observation_id":377873006,"observer":"dinomariobob","bytes":167744,"sha256":"cc5e25afa5041ce2d6ea4e0d726793843f3a867f30b8d9ccf55892d6617da887","ext":"jpg"},{"id":"white_throated_sparrow-17","label":"white_throated_sparrow","photo_id":440986574,"observation_id":246671481,"observer":"suzannehale","bytes":141857,"sha256":"4a354c189225de2e7b4e94a6df9cbd3a671dac0c8a7d2d4c75e933f93d8b83bf","ext":"jpeg"},{"id":"white_throated_sparrow-18","label":"white_throated_sparrow","photo_id":113217820,"observation_id":69733707,"observer":"kemper","bytes":97802,"sha256":"4de7da3ac53a086f2c343555be5a2b62a683715db9f80636a76673cc380fa7f6","ext":"jpeg"},{"id":"white_throated_sparrow-19","label":"white_throated_sparrow","photo_id":575307217,"observation_id":318327478,"observer":"k-simpkins","bytes":90416,"sha256":"af2b4e72a098bd90b920c8a49632e1bbff18b73954d8ac541ca81ed4baf0530b","ext":"jpg"},{"id":"white_throated_sparrow-20","label":"white_throated_sparrow","photo_id":340420076,"observation_id":193765555,"observer":"don54","bytes":62714,"sha256":"7f6eea434fc700166af1d343951d15f3a9c83eff06cfb0518c3c4291019a5556","ext":"jpeg"},{"id":"white_throated_sparrow-21","label":"white_throated_sparrow","photo_id":562344160,"observation_id":311510652,"observer":"rrfc","bytes":45842,"sha256":"f7c6394649765db6e3313a03ae013291db1ae20d89d3b8f0bdf6641757f31ed0","ext":"jpg"},{"id":"white_throated_sparrow-22","label":"white_throated_sparrow","photo_id":74598266,"observation_id":47039878,"observer":"ianrwhyte","bytes":141502,"sha256":"b0633999572a4499a80310955ab928a86f4fe08774e669b4c4949cd26254428b","ext":"jpeg"},{"id":"white_throated_sparrow-23","label":"white_throated_sparrow","photo_id":588001234,"observation_id":324825124,"observer":"portablecity","bytes":370187,"sha256":"11e3a97f43de5d61266d15028fe9023eef3900cfea2d027bd94ad847ecba9607","ext":"jpg"},{"id":"white_throated_sparrow-24","label":"white_throated_sparrow","photo_id":694103481,"observation_id":379467387,"observer":"memoosborne","bytes":120656,"sha256":"82684f82f62f6e036e206eec315fff8e1ba36d308b667eba06750b75d1fada3a","ext":"jpg"},{"id":"white_throated_sparrow-25","label":"white_throated_sparrow","photo_id":655749108,"observation_id":359408087,"observer":"russnamitz","bytes":58449,"sha256":"2a7d2b4226f1941d0950adc04bfd8fe62345f1138c521d0f5f0adf8bc5838da1","ext":"jpg"},{"id":"white_throated_sparrow-26","label":"white_throated_sparrow","photo_id":599110899,"observation_id":330395670,"observer":"jd_flores","bytes":121343,"sha256":"5eeef4f5533a4b0a0222dcf7be000a2b34f3331f3be2b939703155ba8c8ba3ae","ext":"jpg"},{"id":"white_throated_sparrow-27","label":"white_throated_sparrow","photo_id":653888173,"observation_id":358443775,"observer":"toknowtheland","bytes":144197,"sha256":"20a4933b00967fe4488296b2b6e89c12ecbcca0e40da591bd95927cca46ae2f7","ext":"jpg"},{"id":"white_throated_sparrow-28","label":"white_throated_sparrow","photo_id":295037357,"observation_id":170132018,"observer":"sturuss","bytes":117077,"sha256":"d309f75ad5f4008c906ba7b3d6138aa017f919a0a17a1d5b431fa7ab47d5f2d3","ext":"jpg"},{"id":"white_throated_sparrow-29","label":"white_throated_sparrow","photo_id":405042885,"observation_id":228270930,"observer":"carterdorscht","bytes":122157,"sha256":"f1fb32bcfb78f66f50ccd20f2418832afe72c7a5847bdeb8e4dc19546455cfaf","ext":"jpeg"},{"id":"dark_eyed_junco-00","label":"dark_eyed_junco","photo_id":172110799,"observation_id":102901486,"observer":"schylerbrown","bytes":182973,"sha256":"185209c7a1111fc626a068e136ec3cfcdff3174d15f1c60af209d7b32fe7bef9","ext":"jpeg"},{"id":"dark_eyed_junco-01","label":"dark_eyed_junco","photo_id":46691943,"observation_id":29901256,"observer":"haida_gwaii","bytes":46823,"sha256":"a92dca21e6e58c375fc313f0d1da2c86c11408beb0df8fd96fc60ae035ae80d5","ext":"jpg"},{"id":"dark_eyed_junco-02","label":"dark_eyed_junco","photo_id":707222551,"observation_id":386266764,"observer":"ben142","bytes":289608,"sha256":"7bf320edf4d34a4848f3e9d175cfafa66cc5bab6a1a8f97c41c670263de3ff89","ext":"jpg"},{"id":"dark_eyed_junco-03","label":"dark_eyed_junco","photo_id":192557376,"observation_id":114006980,"observer":"k-simpkins","bytes":172398,"sha256":"206f012add8a5d0fa434e07c51f1e7bd73a60c4d299a2202163fc662e1b03479","ext":"jpeg"},{"id":"dark_eyed_junco-04","label":"dark_eyed_junco","photo_id":8793471,"observation_id":6892999,"observer":"truthseqr","bytes":45302,"sha256":"f8241eab39797c4e097a1432b13658466287d61ed515448457907c17e60cf5e2","ext":"jpeg"},{"id":"dark_eyed_junco-05","label":"dark_eyed_junco","photo_id":346777340,"observation_id":196961623,"observer":"zacharyfoster","bytes":46712,"sha256":"ea8f9f0eebb4d2f86193c705344a8fab09bf634bc2217a0874ee57c4f0f5b4ab","ext":"jpg"},{"id":"dark_eyed_junco-06","label":"dark_eyed_junco","photo_id":274980085,"observation_id":159160633,"observer":"andy71","bytes":51209,"sha256":"c54b45ca7fdc635bdb31eb89166c9fce84f0b0f8f0c17331fd5e42b582633c9b","ext":"jpeg"},{"id":"dark_eyed_junco-07","label":"dark_eyed_junco","photo_id":243303909,"observation_id":141959574,"observer":"andywilson","bytes":71321,"sha256":"ee5308b6f93fb40a6d795a7d8ca6f2ef55b4844513f4268ef34827e1c5e4c4a6","ext":"jpeg"},{"id":"dark_eyed_junco-08","label":"dark_eyed_junco","photo_id":213798376,"observation_id":126031618,"observer":"nathanael15","bytes":57646,"sha256":"d6b9e7d2dcc63cdd88b47cce328149aa9e8eb486ee2a5fc0f08b90601f2c7d9b","ext":"jpg"},{"id":"dark_eyed_junco-09","label":"dark_eyed_junco","photo_id":63482066,"observation_id":39977347,"observer":"chrisleearm","bytes":41870,"sha256":"90573e814dbde965c70a934d9702110c40670aa22cc9f80ff8849db877d1fd72","ext":"jpeg"},{"id":"dark_eyed_junco-10","label":"dark_eyed_junco","photo_id":458710472,"observation_id":255914048,"observer":"joy4birds","bytes":90973,"sha256":"fd25ecb1bf86896b84e9e4e8329c742011011e6883631f47dbe3744ba7463153","ext":"jpg"},{"id":"dark_eyed_junco-11","label":"dark_eyed_junco","photo_id":20784016,"observation_id":14046286,"observer":"gambolingquail","bytes":93957,"sha256":"a454d6f987153317c9c57c05019b08ab0ebebc46d3cf72356b014231b69a1e9d","ext":"jpeg"},{"id":"dark_eyed_junco-12","label":"dark_eyed_junco","photo_id":332842159,"observation_id":189933284,"observer":"jan-konilu","bytes":110145,"sha256":"d18706f536164a72a4ec9bacf47437edd89d6b547f86eefdf10ed0166cebf0dd","ext":"jpeg"},{"id":"dark_eyed_junco-13","label":"dark_eyed_junco","photo_id":513004508,"observation_id":270404136,"observer":"thevertebratepokedex","bytes":141252,"sha256":"854f75e7527932d0409e14756725388de9913c0b450536da717e54d683de169f","ext":"jpg"},{"id":"dark_eyed_junco-14","label":"dark_eyed_junco","photo_id":593499256,"observation_id":327583635,"observer":"orionid","bytes":108583,"sha256":"7eab4cea6d46faa878732d82c5c2ece63ecc3e66781ea08d0bd950eee581d16d","ext":"jpg"},{"id":"dark_eyed_junco-15","label":"dark_eyed_junco","photo_id":63590391,"observation_id":40041059,"observer":"bobbyblackmore","bytes":82853,"sha256":"87746be0bca30fc40ddba739c1a35ed1fdd41882ebfe3561518b69d285f5ae5d","ext":"jpeg"},{"id":"dark_eyed_junco-16","label":"dark_eyed_junco","photo_id":12580989,"observation_id":9282523,"observer":"artemis224","bytes":216253,"sha256":"15198c123c01fa0f8c03edc086408c8d95ffa553083c380e8ae236dbb7056093","ext":"jpg"},{"id":"dark_eyed_junco-17","label":"dark_eyed_junco","photo_id":12533281,"observation_id":9255403,"observer":"braincellsgone","bytes":40857,"sha256":"be34c51655f59612858d888b05a4f927da65e1aede850c9a5f1be68fa00bfcbe","ext":"jpg"},{"id":"dark_eyed_junco-18","label":"dark_eyed_junco","photo_id":256948665,"observation_id":149140989,"observer":"igor322","bytes":86925,"sha256":"9339b8cf4aa347b4eb176ef1209dfaa4fcb11e277889cc17de399d6af2c8633a","ext":"jpeg"},{"id":"dark_eyed_junco-19","label":"dark_eyed_junco","photo_id":106718005,"observation_id":66230973,"observer":"vicki936","bytes":41475,"sha256":"e6903e9a69e987c45edd468ace0bf71adf8252063ff5d39c2130409d8fea68be","ext":"jpeg"},{"id":"dark_eyed_junco-20","label":"dark_eyed_junco","photo_id":611049594,"observation_id":336277421,"observer":"skylar_schell","bytes":15425,"sha256":"7e876479febb3d44b1e494a25f778efe4e8bfd323279031cb08f9f6fe95f742e","ext":"jpg"},{"id":"dark_eyed_junco-21","label":"dark_eyed_junco","photo_id":591147962,"observation_id":326403963,"observer":"toknowtheland","bytes":106373,"sha256":"07770f3313dc969802355fa8b3e62a86edb115a64771bd38938e1fab33220c39","ext":"jpg"},{"id":"dark_eyed_junco-22","label":"dark_eyed_junco","photo_id":469746672,"observation_id":261469406,"observer":"shannon_j","bytes":74167,"sha256":"ec4c7ce15aa1c4eadd3bc5d46d433baf9560d6a0e4593d8d57aab8fa9bec9e18","ext":"jpg"},{"id":"dark_eyed_junco-23","label":"dark_eyed_junco","photo_id":459696953,"observation_id":256398190,"observer":"w_mark_c","bytes":262126,"sha256":"5025e757c026e02160ff3f0e82f1f4434b888a6c9689e7ac76274c1fcec64f0c","ext":"jpg"},{"id":"dark_eyed_junco-24","label":"dark_eyed_junco","photo_id":484171775,"observation_id":269292238,"observer":"aschuman","bytes":59459,"sha256":"89dd933d12cf21c0e73eb01e96a9279e2e57b36676a72a9c860281d0d7703282","ext":"jpg"},{"id":"dark_eyed_junco-25","label":"dark_eyed_junco","photo_id":357382685,"observation_id":202362003,"observer":"dougbrown","bytes":56324,"sha256":"70275d8e07192c99e121b67d206de6823b194f043e65b9766f1f9ce772953c78","ext":"jpeg"},{"id":"dark_eyed_junco-26","label":"dark_eyed_junco","photo_id":7660371,"observation_id":6119391,"observer":"jeffreyleeisanaturalist","bytes":108394,"sha256":"0f667e9af8e8d3585807da58a2f315c5dbcb7eece5f6862006fb9da68a4bef42","ext":"jpg"},{"id":"dark_eyed_junco-27","label":"dark_eyed_junco","photo_id":437957476,"observation_id":245430473,"observer":"eug302","bytes":106520,"sha256":"394abfbcf192ecf5a76cb2f19dfa2a62070fa3874b084d1468219d6ee5db6cbf","ext":"jpeg"},{"id":"dark_eyed_junco-28","label":"dark_eyed_junco","photo_id":339439778,"observation_id":193239701,"observer":"rawcomposition","bytes":32258,"sha256":"73fad57ed975df01c529e77eaca9eab9b00741ea2cfc91358856aefca620a10a","ext":"jpg"},{"id":"dark_eyed_junco-29","label":"dark_eyed_junco","photo_id":6198359,"observation_id":5055484,"observer":"glmory","bytes":108168,"sha256":"d31767f42eaaf7f3527133fffb9a0271a9dbbe66fc036d5c694b605563d09965","ext":"jpg"},{"id":"house_finch-00","label":"house_finch","photo_id":117990649,"observation_id":72375345,"observer":"kristen163","bytes":75945,"sha256":"eeafad0dd2e91ecfe45c9d1f27dd0392a01bd81099549c60fd2e36a4b4342a9f","ext":"jpeg"},{"id":"house_finch-01","label":"house_finch","photo_id":389479656,"observation_id":220010434,"observer":"aster-asti","bytes":82128,"sha256":"a8848197b4e7890e07538d492480c4275b75d04e10c1ae95aee91aaafe3319c5","ext":"jpg"},{"id":"house_finch-02","label":"house_finch","photo_id":176982307,"observation_id":105476125,"observer":"vicki936","bytes":22211,"sha256":"c377fb361df0324c7a856d9344968886ece3b94bd67188c9325b8d2d284d3a2f","ext":"jpeg"},{"id":"house_finch-03","label":"house_finch","photo_id":697940852,"observation_id":381438133,"observer":"ben142","bytes":196735,"sha256":"a89f8e0263fdabb404b462acaa592f5dd2ac88ee4615da444470de4a1fae82d5","ext":"jpg"},{"id":"house_finch-04","label":"house_finch","photo_id":72470599,"observation_id":45698380,"observer":"henrya","bytes":61724,"sha256":"1b96d37a7078e1b725b80af4b10848da58b0d0c17a70c8ac01e326c0a749ee6b","ext":"jpeg"},{"id":"house_finch-05","label":"house_finch","photo_id":98576538,"observation_id":61594129,"observer":"enspring","bytes":46784,"sha256":"8b355426d8fe6327f202c6a9458cee1b95de445bfbf7e48a4b5a2c7d0eb78a8c","ext":"jpg"},{"id":"house_finch-06","label":"house_finch","photo_id":80751781,"observation_id":50842166,"observer":"leahmfulton","bytes":44269,"sha256":"6d6202de26f042d83ee6c5af550cd74e6ab10eb796eed2f78c83ac9e2368e4b7","ext":"jpg"},{"id":"house_finch-07","label":"house_finch","photo_id":630196420,"observation_id":345777550,"observer":"truthseqr","bytes":149455,"sha256":"abb84d1e327dd82c07cbea3dd5583c07453e69b2cc220397b101e597da81bd6c","ext":"jpg"},{"id":"house_finch-08","label":"house_finch","photo_id":214612538,"observation_id":126483167,"observer":"hamiltonturner","bytes":124116,"sha256":"6b7687640c4641b974865da04cf9eaf1f86b774ebc19678f2fc39e55c8648930","ext":"jpeg"},{"id":"house_finch-09","label":"house_finch","photo_id":213077180,"observation_id":125637342,"observer":"jnicat","bytes":25823,"sha256":"e1e3baff8d0bd72339e3e49089f7f4f2c1dd383ff005e49a964a1bacc4f8ebb8","ext":"jpeg"},{"id":"house_finch-10","label":"house_finch","photo_id":500744872,"observation_id":278868417,"observer":"pbaff","bytes":149626,"sha256":"2684cfb1fc40d7766610a5920ead0ad0c27c338ccb4fb9ed18baddda150568b0","ext":"jpeg"},{"id":"house_finch-11","label":"house_finch","photo_id":268678834,"observation_id":155431721,"observer":"stevestevens","bytes":120325,"sha256":"5d1e8c097c214d14ef7b895bf95769ae9bc25fa799a98bbfa6f12f2f84e6af79","ext":"jpeg"},{"id":"house_finch-12","label":"house_finch","photo_id":358373136,"observation_id":202884575,"observer":"kcthetc1","bytes":52329,"sha256":"b9929163e4fdadef26c753437ac7af3ad550aa047b15cba131c05d4d335799fe","ext":"jpeg"},{"id":"house_finch-13","label":"house_finch","photo_id":104227663,"observation_id":64793290,"observer":"verdantpulsar","bytes":101003,"sha256":"90fe37c477bad9ed30ab119e1a7445ffc2720e548a22bb65d66b2ff7b82337d5","ext":"jpeg"},{"id":"house_finch-14","label":"house_finch","photo_id":196156834,"observation_id":116218899,"observer":"kgarrett","bytes":56801,"sha256":"ce03d1d70a89b5e6e0088f4307f8077573c3571b91997725ca2e6c69be80e2d0","ext":"jpeg"},{"id":"house_finch-15","label":"house_finch","photo_id":454332148,"observation_id":253709711,"observer":"rlaortiz","bytes":149985,"sha256":"cf007ef8ac57bc0eb085c9eecf9fc95eb69a29df706998de237df24f9491c61d","ext":"jpeg"},{"id":"house_finch-16","label":"house_finch","photo_id":509969159,"observation_id":283798927,"observer":"damienxw","bytes":104277,"sha256":"56d3656be473c362f1ccd09d15e62cbcfe9137d83bef7a3312d72444921c7805","ext":"jpg"},{"id":"house_finch-17","label":"house_finch","photo_id":665287910,"observation_id":295246528,"observer":"dinomariobob","bytes":74972,"sha256":"5f6121c1f8dbfaccbb61c279a579c22600744eb4118c2eeef09e69c86f6e1a49","ext":"jpg"},{"id":"house_finch-18","label":"house_finch","photo_id":168402062,"observation_id":100871757,"observer":"k-simpkins","bytes":86922,"sha256":"209a884cf7618d0b85679ae3a72f237a6d03a5a3096f6ab1b2e5503c38c50d9d","ext":"jpg"},{"id":"house_finch-19","label":"house_finch","photo_id":247557547,"observation_id":144258909,"observer":"aparrot1","bytes":79152,"sha256":"115302ef807e6f4d132df584a20ba8644846f05dfe9fee27a7c0e07ec89d87b7","ext":"jpg"},{"id":"house_finch-20","label":"house_finch","photo_id":249874672,"observation_id":145517234,"observer":"vijaybarve","bytes":115757,"sha256":"15538eeeb228d284f49f33d0bda77626b57fdaabe90d05f4379ffc3bbd855703","ext":"jpeg"},{"id":"house_finch-21","label":"house_finch","photo_id":379972789,"observation_id":214941038,"observer":"dougbrown","bytes":29723,"sha256":"d7fea293bd3f92aec7be52bdfe5b904793c53cfaeacf9ffd2e762702ee10ba74","ext":"jpeg"},{"id":"house_finch-22","label":"house_finch","photo_id":250140068,"observation_id":145607589,"observer":"matthias55","bytes":87579,"sha256":"62adc5d86cbdb613779fefa84f6f74f306b6fab5bfb86320f4ef40cd2571ff6a","ext":"jpeg"},{"id":"house_finch-23","label":"house_finch","photo_id":247757548,"observation_id":144364266,"observer":"nana10","bytes":123890,"sha256":"10b3b6c0b4273a31597050df4218899ddcbe31cdfa60d9a46421ed0bf3d26558","ext":"jpeg"},{"id":"house_finch-24","label":"house_finch","photo_id":253927599,"observation_id":147557306,"observer":"kerykeion","bytes":91321,"sha256":"6166a9bf59e2075a1129b344a24f3fcafccb610eb984510c5b4c3f9b3abe96af","ext":"jpeg"},{"id":"house_finch-25","label":"house_finch","photo_id":250651171,"observation_id":145869416,"observer":"cathartic_cathartes","bytes":55197,"sha256":"1929f770ec5ca766c6b88ffdbad5fd7c27df033e12fe46109fd87dc1c835fc59","ext":"jpeg"},{"id":"house_finch-26","label":"house_finch","photo_id":244648136,"observation_id":142674679,"observer":"michelle_lopez","bytes":51377,"sha256":"9422e42758a68c343d0487d07726819424a24c8271a94aa48c75f4c1e339be77","ext":"jpg"},{"id":"house_finch-27","label":"house_finch","photo_id":373687946,"observation_id":211229903,"observer":"c_dizzy","bytes":145810,"sha256":"ae7289dc681f8e192886d47018ee70aa9586f08e618e57fa1fe195a5695e1779","ext":"jpeg"},{"id":"house_finch-28","label":"house_finch","photo_id":242992511,"observation_id":141794074,"observer":"chrisleearm","bytes":102901,"sha256":"b6bd72b41f04d2b6a7855b28e2b169d5aad6663db4a0ac0d364b22ffc2512018","ext":"jpg"},{"id":"house_finch-29","label":"house_finch","photo_id":110518705,"observation_id":68278832,"observer":"kemper","bytes":103151,"sha256":"62ee726242d1970d9bb8536e31c06635afc92b9352d1394574acb1d9d5eb26ed","ext":"jpeg"},{"id":"american_goldfinch-00","label":"american_goldfinch","photo_id":84579952,"observation_id":53187208,"observer":"glennberry","bytes":59673,"sha256":"72d36079e592e0a83c2f774f9073bfd4cc81253452c925d1673217ddd4b52a36","ext":"jpeg"},{"id":"american_goldfinch-01","label":"american_goldfinch","photo_id":12533322,"observation_id":9255418,"observer":"braincellsgone","bytes":55661,"sha256":"6344e0125e74791f43ac6e07e5e1b9fbfce6d19bc62b6bb5d83b3caff9f7bcbc","ext":"jpg"},{"id":"american_goldfinch-02","label":"american_goldfinch","photo_id":131102823,"observation_id":80016788,"observer":"radrat","bytes":98990,"sha256":"232a944f7e3351d4916a12ef2f6d598e7b007caaa95a9b064a14c1ba3af2a6ff","ext":"jpeg"},{"id":"american_goldfinch-03","label":"american_goldfinch","photo_id":175048222,"observation_id":104466897,"observer":"eug302","bytes":44231,"sha256":"11c723482cc75fcc3a723ac1c0818a68e2fcf74c4ca684cf60195bf0d33f274e","ext":"jpg"},{"id":"american_goldfinch-04","label":"american_goldfinch","photo_id":68849595,"observation_id":43390778,"observer":"mefisher","bytes":154503,"sha256":"66f07bc59bb3fdedd65a4537ebabd0cafd457826b8bf4bb633181f584a3edfd1","ext":"jpg"},{"id":"american_goldfinch-05","label":"american_goldfinch","photo_id":431916465,"observation_id":242278180,"observer":"k-simpkins","bytes":45413,"sha256":"d76e7adf33e3a84ebec24dde5438965e62ebec8595755453973846340e4f460d","ext":"jpg"},{"id":"american_goldfinch-06","label":"american_goldfinch","photo_id":377136648,"observation_id":213398931,"observer":"nathan1177","bytes":66516,"sha256":"7ad75838fdf2020a8e426e97507c7dd4355da93e6eece241128c28adbe302438","ext":"jpg"},{"id":"american_goldfinch-07","label":"american_goldfinch","photo_id":230801384,"observation_id":135330401,"observer":"enspring","bytes":42886,"sha256":"78aebfb9b28c3e16dd9618a0e1ae06df67bfa4b8550c0879427d96915c475fed","ext":"jpeg"},{"id":"american_goldfinch-08","label":"american_goldfinch","photo_id":660472044,"observation_id":361884286,"observer":"ben142","bytes":270599,"sha256":"733d64cd50c61334682f0862f5c7859ded34cae5776ee0bc6594fee400dc4876","ext":"jpg"},{"id":"american_goldfinch-09","label":"american_goldfinch","photo_id":294667312,"observation_id":169935316,"observer":"dande","bytes":163470,"sha256":"39e7892e81eeef6af4887e61bc0998e17797688eb6394fcc8c9438391e875ee0","ext":"jpeg"},{"id":"american_goldfinch-10","label":"american_goldfinch","photo_id":143215717,"observation_id":86889530,"observer":"memoosborne","bytes":129438,"sha256":"ef4a9a771cef9ba3c2c047eb106a6aa220236dd6aaa6aade5f4ef3a37c8abcba","ext":"jpeg"},{"id":"american_goldfinch-11","label":"american_goldfinch","photo_id":403873164,"observation_id":227647158,"observer":"drew_baxter","bytes":106009,"sha256":"74e34c776f1b9a5d375a7dfae0e309cd42dbd133b82a1ecad4a49111ac7eed49","ext":"jpeg"},{"id":"american_goldfinch-12","label":"american_goldfinch","photo_id":481153019,"observation_id":267722510,"observer":"vicki936","bytes":255397,"sha256":"3326ddbee3270241b681cb636466e742491f110e9841f453f5158864aadaea36","ext":"jpg"},{"id":"american_goldfinch-13","label":"american_goldfinch","photo_id":213311122,"observation_id":125763980,"observer":"hickl","bytes":24740,"sha256":"0c295b3761bced4215519ff24a4f734773b54be98de456a4444d0819456af7dd","ext":"jpeg"},{"id":"american_goldfinch-14","label":"american_goldfinch","photo_id":417352824,"observation_id":234736320,"observer":"joy4birds","bytes":81109,"sha256":"357014c108519543471b94f39591667d1a67d87fd1fdc4702a3a449235d6bddd","ext":"jpeg"},{"id":"american_goldfinch-15","label":"american_goldfinch","photo_id":72130720,"observation_id":45486482,"observer":"dctphoto","bytes":178111,"sha256":"17b2f3599e20161acc17fd63bf61e9f40488b9c4d5bfaefc8cae54de42997509","ext":"jpeg"},{"id":"american_goldfinch-16","label":"american_goldfinch","photo_id":45148898,"observation_id":28952026,"observer":"megachile","bytes":49358,"sha256":"6bbc6ca0ac074e486c20ce4c3fad5dc863cb2e908efc2b1ef97494403a351252","ext":"jpeg"},{"id":"american_goldfinch-17","label":"american_goldfinch","photo_id":133251099,"observation_id":81250513,"observer":"raffib128","bytes":128110,"sha256":"863c76587fe1de80a84b97c2e72f38ae1cf4fa972789e006d9b0b544d696c6ef","ext":"jpeg"},{"id":"american_goldfinch-18","label":"american_goldfinch","photo_id":155797129,"observation_id":93957328,"observer":"nathanael15","bytes":74086,"sha256":"7059ae3bdeeb48fe949e5b70b788bd28c96aecd0fe49cb7be22db9a8697bf5a5","ext":"jpg"},{"id":"american_goldfinch-19","label":"american_goldfinch","photo_id":11400438,"observation_id":8535447,"observer":"akneidel","bytes":26423,"sha256":"4af74c1d04ddc7bbb7bb0e9eb2977a1daf48dd9b9477d71d69a7c4cb5d4f785b","ext":"jpg"},{"id":"american_goldfinch-20","label":"american_goldfinch","photo_id":145464774,"observation_id":88186208,"observer":"wildreturn","bytes":68204,"sha256":"565d2a3b6d404e9ea0c24737ebcc5a3a82057b1452b4790df5e9552e8c50e592","ext":"jpg"},{"id":"american_goldfinch-21","label":"american_goldfinch","photo_id":55990791,"observation_id":35505213,"observer":"conhawn","bytes":84915,"sha256":"69f768c39a2180440bdcbfc6addc5d426341e8080d4cf9ba8241d57564b3e6fb","ext":"jpg"},{"id":"american_goldfinch-22","label":"american_goldfinch","photo_id":637247336,"observation_id":289067166,"observer":"dinomariobob","bytes":142312,"sha256":"01c63d305fce8edcc3a494f0543154c6bd6aa52368e02be84cf6c23e95b94cdd","ext":"jpg"},{"id":"american_goldfinch-23","label":"american_goldfinch","photo_id":460988055,"observation_id":257029007,"observer":"eric112","bytes":60314,"sha256":"0db1b5f0e32d1faf861a437a20794fa18e80ea8966313933c145451c9319e2a9","ext":"jpg"},{"id":"american_goldfinch-24","label":"american_goldfinch","photo_id":59072170,"observation_id":37280587,"observer":"bradenjudson","bytes":23468,"sha256":"827d77bf9ca9cc456e867434a07b68cdceb4a20e09eb9e6b67757c80cd31ff42","ext":"jpeg"},{"id":"american_goldfinch-25","label":"american_goldfinch","photo_id":109875643,"observation_id":67928198,"observer":"artemis224","bytes":217909,"sha256":"99e1d7d30eb19d47c7974e9ddd7efe4d06329c952a41e15d0a4b2095b747df61","ext":"jpg"},{"id":"american_goldfinch-26","label":"american_goldfinch","photo_id":66515260,"observation_id":41915252,"observer":"reuvenm","bytes":58833,"sha256":"90f0b687d9626fdf5d0111b794cf960cf18f3de4c8eab601fcb23c41b45c4df7","ext":"jpeg"},{"id":"american_goldfinch-27","label":"american_goldfinch","photo_id":112715850,"observation_id":69464521,"observer":"umamimomma","bytes":78717,"sha256":"fae2edcb0901dada461a9ac6b3873d6479e8faf66bfd26a0ad47021f6bcff704","ext":"jpg"},{"id":"american_goldfinch-28","label":"american_goldfinch","photo_id":123422249,"observation_id":75440388,"observer":"rachel_bosley","bytes":131792,"sha256":"864be3aebb7f1c8c9ed01cdf2e16b690f11354bb37e53b23d44671082afdbcfc","ext":"jpg"},{"id":"american_goldfinch-29","label":"american_goldfinch","photo_id":252964992,"observation_id":147051902,"observer":"carterdorscht","bytes":149077,"sha256":"a8360d1774df42f034692863781af47fd030265df9bcc1fc938333fabc673c9e","ext":"jpeg"}]

SPECIES = {
    "song_sparrow": "Song Sparrow",
    "chipping_sparrow": "Chipping Sparrow",
    "white_throated_sparrow": "White-throated Sparrow",
    "dark_eyed_junco": "Dark-eyed Junco",
    "house_finch": "House Finch",
    "american_goldfinch": "American Goldfinch",
}
print({"records": len(SAMPLE_RECORDS), "classes": len(SPECIES), "license": CORPUS_LICENSE})


In [ ]:
# @title 2.2 Acquire, validate, and split the dataset
MAX_ARCHIVE_EXPANDED_BYTES = 1 * 1024**3

def sha256_bytes(payload):
    return hashlib.sha256(payload).hexdigest()

def safe_extract_zip(path, destination):
    destination.mkdir(parents=True, exist_ok=True)
    seen = set()
    expanded = 0
    with zipfile.ZipFile(path) as archive:
        for info in archive.infolist():
            member = PurePosixPath(info.filename)
            if member.is_absolute() or ".." in member.parts:
                raise ValueError(f"Unsafe archive path: {info.filename!r}")
            if info.filename in seen:
                raise ValueError(f"Duplicate archive member: {info.filename!r}")
            seen.add(info.filename)
            mode = (info.external_attr >> 16) & 0o170000
            if mode == stat.S_IFLNK:
                raise ValueError(f"Symlink entries are not allowed: {info.filename!r}")
            expanded += info.file_size
            if expanded > MAX_ARCHIVE_EXPANDED_BYTES:
                raise ValueError("Archive expands beyond the 1 GiB workshop ceiling.")
        archive.extractall(destination)
    return destination

def validate_manifest(frame, root):
    required = ["id", "file", "label", "split"]
    missing = [c for c in required if c not in frame.columns]
    if missing:
        raise ValueError(f"labels.csv is missing required columns: {missing}")
    if frame.columns.duplicated().any():
        raise ValueError("labels.csv contains duplicate column names.")
    if frame["id"].duplicated().any():
        raise ValueError("labels.csv id values must be unique.")
    allowed = {"train", "validation", "test"}
    if set(frame["split"]) - allowed:
        raise ValueError("split must contain only train, validation, or test.")
    labels = sorted(frame["label"].astype(str).unique())
    if len(labels) < 2 or len(labels) > 100:
        raise ValueError(f"Expected 2..100 classes; got {len(labels)}.")
    resolved_root = root.resolve()
    hashes = {}
    for row in frame.itertuples(index=False):
        rel = PurePosixPath(str(row.file))
        if rel.is_absolute() or ".." in rel.parts:
            raise ValueError(f"Unsafe relative image path: {row.file!r}")
        path = (root / Path(*rel.parts)).resolve()
        if resolved_root not in path.parents and path != resolved_root:
            raise ValueError(f"Image escapes dataset root: {row.file!r}")
        if not path.is_file():
            raise ValueError(f"Image file not found: {row.file!r}")
        digest = hashlib.sha256(path.read_bytes()).hexdigest()
        if digest in hashes:
            raise ValueError(f"Duplicate image bytes detected: {row.file!r} and {hashes[digest]!r}")
        hashes[digest] = row.file
    split_classes = frame.groupby("split")["label"].apply(lambda x: set(map(str, x)))
    expected = set(labels)
    for split in ("train", "validation", "test"):
        if split not in split_classes or split_classes[split] != expected:
            raise ValueError(f"{split} must contain every class; got {sorted(split_classes.get(split, set()))}.")
    return labels

if USE_BYOD:
    if not BYOD_ZIP_PATH:
        raise ValueError("USE_BYOD=True requires BYOD_ZIP_PATH pointing to a local ZIP file.")
    archive_path = Path(BYOD_ZIP_PATH)
    if not archive_path.is_file():
        raise FileNotFoundError(archive_path)
    dataset_root = safe_extract_zip(archive_path, DATA_ROOT / "byod")
    labels_path = dataset_root / "labels.csv"
    if not labels_path.is_file():
        candidates = list(dataset_root.rglob("labels.csv"))
        if len(candidates) != 1:
            raise ValueError("BYOD ZIP must contain exactly one labels.csv.")
        labels_path = candidates[0]
        dataset_root = labels_path.parent
    dataset_manifest = pd.read_csv(labels_path)
    class_names = validate_manifest(dataset_manifest, dataset_root)
    sample_kind = "BYOD"
    dataset_sha256 = hashlib.sha256(archive_path.read_bytes()).hexdigest()
else:
    dataset_root = DATA_ROOT / "inat_birds"
    image_root = dataset_root / "images"
    image_root.mkdir(parents=True, exist_ok=True)
    materialized = []
    for item in SAMPLE_RECORDS:
        output = image_root / f"{item['photo_id']}.{item['ext']}"
        payload = output.read_bytes() if output.is_file() else b""
        if len(payload) != item["bytes"] or sha256_bytes(payload) != item["sha256"]:
            url = f"{CORPUS_BASE_URL}{item['photo_id']}/medium.{item['ext']}"
            request = urllib.request.Request(url, headers={"User-Agent": "dimer-image-workshop/2.1"})
            with urllib.request.urlopen(request, timeout=120) as response:
                payload = response.read()
            if len(payload) != item["bytes"] or sha256_bytes(payload) != item["sha256"]:
                raise ValueError(f"Pinned image verification failed for {item['id']}")
            output.write_bytes(payload)
        materialized.append({**item, "file": str(Path("images") / output.name)})

    rng = random.Random(SAMPLE_SEED)
    rows = []
    for label in sorted(SPECIES):
        pool = [item.copy() for item in materialized if item["label"] == label]
        rng.shuffle(pool)
        cursor = 0
        for split, per_class in SAMPLE_SPLIT.items():
            for item in pool[cursor:cursor + per_class]:
                rows.append({
                    "id": item["id"],
                    "file": item["file"],
                    "label": item["label"],
                    "split": split,
                    "observer": item["observer"],
                    "observation_id": item["observation_id"],
                    "sha256": item["sha256"],
                })
            cursor += per_class
    dataset_manifest = pd.DataFrame(rows)
    class_names = validate_manifest(dataset_manifest, dataset_root)
    dataset_manifest.to_csv(dataset_root / "labels.csv", index=False)
    manifest_bytes = dataset_manifest.sort_values("id").to_csv(index=False, lineterminator="\n").encode()
    dataset_sha256 = hashlib.sha256(manifest_bytes).hexdigest()
    sample_kind = "pinned iNaturalist CC0 sample"

print({
    "sample_kind": sample_kind,
    "rows": len(dataset_manifest),
    "classes": class_names,
    "split_counts": dataset_manifest["split"].value_counts().to_dict(),
    "manifest_sha256": dataset_sha256,
})
display(dataset_manifest.head())


## 3. Common adaptation and evaluation protocol

Every model is treated identically:

1. load the pinned pretrained backbone;
2. resolve that checkpoint's own evaluation resize/crop/normalization;
3. remove the original ImageNet classifier by constructing the model with `num_classes=0`;
4. extract one frozen feature vector per image;
5. L2-normalize those vectors;
6. train a new linear six-class head on **train only**;
7. choose the head checkpoint with highest validation macro-F1;
8. export the selected head as SafeTensors plus a JSON manifest;
9. fresh-reload the head and verify test logits; and
10. only after validation selection is frozen, report the independent test results.

This is **gradient adaptation of the new head only**. No backbone weight is updated. The method is intentionally standardized so architectural differences are not confounded with six different fine-tuning recipes.


In [ ]:
# @title 3.1 Install the isolated comparative runtime
RUNTIME_PINS = [
    "torch==2.14.0",
    "torchvision==0.29.0",
    "torchaudio==2.11.0",
    "timm==1.0.29",
    "huggingface-hub==0.36.2",
    "safetensors==0.8.0",
    "numpy==2.5.3",
    "pandas==3.0.5",
    "pillow==11.3.0",
]
ENV_DIR = RUNTIME_ROOT / "vision_env"
PYTHON = ENV_DIR / ("Scripts/python.exe" if os.name == "nt" else "bin/python")
marker = ENV_DIR / "dimer_pins.json"
pins_text = json.dumps(RUNTIME_PINS, sort_keys=True)

if not PYTHON.exists():
    venv.EnvBuilder(with_pip=True).create(ENV_DIR)
if not marker.exists() or marker.read_text() != pins_text:
    subprocess.run(
        [str(PYTHON), "-m", "pip", "install", "--disable-pip-version-check", "-q", *RUNTIME_PINS],
        check=True,
    )
    marker.write_text(pins_text)

print({"runtime_python": str(PYTHON), "pins": RUNTIME_PINS})


In [ ]:
# @title 3.2 Materialize the standalone model runner
RUNNER_SOURCE = r"""
import csv
import hashlib
import importlib.metadata
import json
import math
import platform
import random
import sys
import time
from pathlib import Path

import numpy as np
import torch
import timm
from huggingface_hub import HfApi
from PIL import Image
from safetensors.torch import load_file, save_file
from timm.data import create_transform, resolve_model_data_config
from torch.utils.data import DataLoader, Dataset

cfg = json.loads(Path(sys.argv[1]).read_text())
out_path = Path(sys.argv[2])
dataset_root = Path(cfg["dataset_root"])
rows = list(csv.DictReader(Path(cfg["manifest_csv"]).open(encoding="utf-8")))
classes = list(cfg["class_names"])
class_to_idx = {name: i for i, name in enumerate(classes)}
device = "cuda" if torch.cuda.is_available() else "cpu"
if device != "cuda":
    raise RuntimeError("This six-model comparative runner requires CUDA.")

def accuracy_macro_f1(y_true, y_pred, n_classes):
    y_true = np.asarray(y_true, dtype=int)
    y_pred = np.asarray(y_pred, dtype=int)
    accuracy = float(np.mean(y_true == y_pred))
    f1s, recalls = [], {}
    for c in range(n_classes):
        tp = int(np.sum((y_true == c) & (y_pred == c)))
        fp = int(np.sum((y_true != c) & (y_pred == c)))
        fn = int(np.sum((y_true == c) & (y_pred != c)))
        precision = tp / (tp + fp) if tp + fp else 0.0
        recall = tp / (tp + fn) if tp + fn else 0.0
        f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
        f1s.append(f1)
        recalls[classes[c]] = recall
    return {"accuracy": accuracy, "macro_f1": float(np.mean(f1s)), "per_class_recall": recalls}

class ImageRows(Dataset):
    def __init__(self, records, transform):
        self.records = records
        self.transform = transform
    def __len__(self):
        return len(self.records)
    def __getitem__(self, index):
        row = self.records[index]
        path = dataset_root / row["file"]
        with Image.open(path) as image:
            image = image.convert("RGB")
            tensor = self.transform(image)
        return tensor, class_to_idx[row["label"]], row["id"]

def extract(model, transform, records):
    loader = DataLoader(ImageRows(records, transform), batch_size=int(cfg["batch_size"]), shuffle=False, num_workers=2)
    features, labels, ids = [], [], []
    started = time.perf_counter()
    model.eval()
    with torch.inference_mode():
        for images, y, batch_ids in loader:
            x = model(images.to(device))
            if x.ndim != 2:
                x = torch.flatten(x, 1)
            x = torch.nn.functional.normalize(x.float(), dim=1)
            features.append(x.cpu())
            labels.append(y.long())
            ids.extend(batch_ids)
    return torch.cat(features), torch.cat(labels), ids, time.perf_counter() - started

def train_head(train_x, train_y, val_x, val_y, feature_dim):
    torch.manual_seed(int(cfg["seed"]))
    head = torch.nn.Linear(feature_dim, len(classes))
    optimizer = torch.optim.AdamW(head.parameters(), lr=float(cfg["head_lr"]), weight_decay=1e-4)
    best = None
    history = []
    for epoch in range(1, int(cfg["head_epochs"]) + 1):
        head.train()
        optimizer.zero_grad(set_to_none=True)
        logits = head(train_x)
        loss = torch.nn.functional.cross_entropy(logits, train_y)
        loss.backward()
        optimizer.step()
        head.eval()
        with torch.no_grad():
            val_logits = head(val_x)
            val_loss = float(torch.nn.functional.cross_entropy(val_logits, val_y))
            val_pred = val_logits.argmax(dim=1).numpy()
        metrics = accuracy_macro_f1(val_y.numpy(), val_pred, len(classes))
        history.append({"epoch": epoch, "train_loss": float(loss), "val_loss": val_loss, **metrics})
        score = (metrics["macro_f1"], -val_loss)
        if best is None or score > best["score"]:
            best = {
                "score": score,
                "epoch": epoch,
                "state": {k: v.detach().clone() for k, v in head.state_dict().items()},
                "metrics": metrics,
                "val_loss": val_loss,
            }
    head.load_state_dict(best["state"])
    head.eval()
    return head, best, history

results = {}
artifact_root = Path(cfg["artifact_root"])
artifact_root.mkdir(parents=True, exist_ok=True)

for key, spec in cfg["models"].items():
    resolved = HfApi().model_info(repo_id=spec["model_id"], revision=spec["revision"]).sha
    if resolved != spec["revision"]:
        raise RuntimeError(f"Hub revision mismatch for {key}: {resolved}")

    reference = f"hf-hub:{spec['model_id']}@{spec['revision']}"
    started_load = time.perf_counter()
    model = timm.create_model(reference, pretrained=True, num_classes=0).to(device).eval()
    load_seconds = time.perf_counter() - started_load
    data_cfg = resolve_model_data_config(model)
    transform = create_transform(**data_cfg, is_training=False)
    params = sum(p.numel() for p in model.parameters())

    split_rows = {name: [row for row in rows if row["split"] == name] for name in ("train","validation","test")}
    train_x, train_y, train_ids, train_seconds = extract(model, transform, split_rows["train"])
    val_x, val_y, val_ids, val_seconds = extract(model, transform, split_rows["validation"])
    test_x, test_y, test_ids, test_seconds = extract(model, transform, split_rows["test"])
    feature_dim = int(train_x.shape[1])

    started_head = time.perf_counter()
    head, best, history = train_head(train_x, train_y, val_x, val_y, feature_dim)
    head_seconds = time.perf_counter() - started_head

    with torch.no_grad():
        val_logits = head(val_x)
        test_logits = head(test_x)
    val_pred = val_logits.argmax(dim=1).numpy()
    test_pred = test_logits.argmax(dim=1).numpy()
    val_metrics = accuracy_macro_f1(val_y.numpy(), val_pred, len(classes))
    test_metrics = accuracy_macro_f1(test_y.numpy(), test_pred, len(classes))

    out_dir = artifact_root / key
    out_dir.mkdir(parents=True, exist_ok=True)
    adapter_path = out_dir / "head.safetensors"
    save_file({name: tensor.detach().contiguous() for name, tensor in head.state_dict().items()}, str(adapter_path))

    manifest = {
        "format": "dimer-workshop-linear-probe",
        "format_version": 1,
        "base_model": {"id": spec["model_id"], "revision": spec["revision"], "license": spec["license"]},
        "classes": classes,
        "feature_dim": feature_dim,
        "adaptation": {
            "type": "gradient-trained linear head over frozen L2-normalized backbone features",
            "epochs_budget": int(cfg["head_epochs"]),
            "selected_epoch": int(best["epoch"]),
            "optimizer": "AdamW",
            "learning_rate": float(cfg["head_lr"]),
            "weight_decay": 1e-4,
            "seed": int(cfg["seed"]),
            "selection_metric": "validation macro_f1 then validation loss",
        },
        "preprocessing": {
            "input_size": list(data_cfg["input_size"]),
            "mean": list(data_cfg["mean"]),
            "std": list(data_cfg["std"]),
            "interpolation": str(data_cfg.get("interpolation")),
            "crop_pct": float(data_cfg.get("crop_pct", 1.0)),
        },
        "files": {"head.safetensors": {"sha256": hashlib.sha256(adapter_path.read_bytes()).hexdigest(), "bytes": adapter_path.stat().st_size}},
    }
    (out_dir / "manifest.json").write_text(json.dumps(manifest, indent=2))

    # Fresh artifact boundary: construct a new head and load only from files.
    fresh = torch.nn.Linear(feature_dim, len(classes))
    tensors = load_file(str(adapter_path))
    fresh.load_state_dict(tensors)
    fresh.eval()
    with torch.no_grad():
        fresh_test = fresh(test_x)
    max_reload_diff = float(torch.max(torch.abs(fresh_test - test_logits)))
    if max_reload_diff > 1e-7:
        raise RuntimeError(f"{key} fresh-reload logits differ by {max_reload_diff}")

    majority_class = int(torch.bincount(train_y, minlength=len(classes)).argmax())
    majority_pred = np.full(len(test_y), majority_class, dtype=int)
    majority_metrics = accuracy_macro_f1(test_y.numpy(), majority_pred, len(classes))

    results[key] = {
        "display_name": spec["display_name"],
        "family": spec["family"],
        "model_id": spec["model_id"],
        "model_revision": spec["revision"],
        "resolved_revision": resolved,
        "license": spec["license"],
        "parameter_count": int(params),
        "feature_dim": feature_dim,
        "preprocessing": manifest["preprocessing"],
        "selected_epoch": int(best["epoch"]),
        "validation": val_metrics,
        "test": test_metrics,
        "majority_test": majority_metrics,
        "history": history,
        "timing_seconds": {
            "load": load_seconds,
            "train_features": train_seconds,
            "validation_features": val_seconds,
            "test_features": test_seconds,
            "head_training": head_seconds,
        },
        "artifact": {
            "directory": str(out_dir),
            "head_sha256": manifest["files"]["head.safetensors"]["sha256"],
            "reload_max_abs_logit_diff": max_reload_diff,
        },
        "predictions": {
            "validation": [{"id": rid, "truth": classes[int(y)], "prediction": classes[int(p)]} for rid,y,p in zip(val_ids,val_y,val_pred)],
            "test": [{"id": rid, "truth": classes[int(y)], "prediction": classes[int(p)]} for rid,y,p in zip(test_ids,test_y,test_pred)],
        },
    }

    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

payload = {
    "runtime": {
        "python": platform.python_version(),
        "torch": torch.__version__,
        "timm": timm.__version__,
        "numpy": np.__version__,
        "pillow": importlib.metadata.version("pillow"),
        "device": device,
    },
    "results": results,
}
out_path.write_text(json.dumps(payload, indent=2))
"""
runner_path = RUNTIME_ROOT / "classification_runner.py"
runner_path.write_text(RUNNER_SOURCE, encoding="utf-8")

runner_cfg = {
    "dataset_root": str(dataset_root),
    "manifest_csv": str(dataset_root / "labels.csv"),
    "class_names": class_names,
    "models": MODELS,
    "batch_size": BATCH_SIZE,
    "head_epochs": HEAD_EPOCHS,
    "head_lr": HEAD_LR,
    "seed": 42,
    "artifact_root": str(ARTIFACT_ROOT),
}
(RUNTIME_ROOT / "classification_runner_config.json").write_text(json.dumps(runner_cfg, indent=2))
print(runner_path)


## 4. Run the six-model transfer-learning experiment

This stage downloads and loads each pinned backbone one at a time, extracts frozen features for all three splits, trains the small linear head on CPU, exports it, verifies a fresh reload, frees the backbone, and moves to the next model.

Processing one backbone at a time keeps GPU memory bounded. EVA-02 uses 448-pixel input and is expected to be the slowest of the six.


In [ ]:
# @title 4.1 Execute all six live DIMER backbones
result_path = RUNTIME_ROOT / "classification_results.json"
started = time.perf_counter()
run = subprocess.run(
    [
        str(PYTHON),
        str(RUNTIME_ROOT / "classification_runner.py"),
        str(RUNTIME_ROOT / "classification_runner_config.json"),
        str(result_path),
    ],
    text=True,
    capture_output=True,
)
WALL_SECONDS = time.perf_counter() - started
if run.returncode != 0:
    print(run.stdout)
    print(run.stderr)
    raise RuntimeError(f"Classification runner failed with exit code {run.returncode}")

RUN_RESULTS = json.loads(result_path.read_text())
print({"runtime": RUN_RESULTS["runtime"], "wall_seconds": round(WALL_SECONDS, 2)})
display(pd.DataFrame([
    {
        "model": value["display_name"],
        "family": value["family"],
        "parameters_m": value["parameter_count"] / 1e6,
        "feature_dim": value["feature_dim"],
        "input_size": value["preprocessing"]["input_size"],
        "selected_epoch": value["selected_epoch"],
        "validation_macro_f1": value["validation"]["macro_f1"],
    }
    for value in RUN_RESULTS["results"].values()
]).sort_values("validation_macro_f1", ascending=False))


## 5. Freeze model selection on validation data

The workshop chooses the backbone with the highest **validation macro-F1**; validation loss was already used only as a tie-breaker within each model's head training.

Only after that decision is recorded do we assemble the final test comparison. This prevents the test set from becoming another tuning set.


In [ ]:
# @title 5.1 Freeze the choice, then reveal independent test metrics
validation_table = pd.DataFrame([
    {
        "key": key,
        "model": value["display_name"],
        "family": value["family"],
        "accuracy": value["validation"]["accuracy"],
        "macro_f1": value["validation"]["macro_f1"],
    }
    for key, value in RUN_RESULTS["results"].items()
]).sort_values(["macro_f1", "accuracy"], ascending=False)

SELECTED_MODEL_KEY = str(validation_table.iloc[0]["key"])
selection_record = {
    "selection_metric": "validation macro_f1, then validation accuracy",
    "selected_model_key": SELECTED_MODEL_KEY,
    "selected_model": MODELS[SELECTED_MODEL_KEY]["display_name"],
    "test_metrics_used_for_selection": False,
}
print(json.dumps(selection_record, indent=2))

test_rows = []
for key, value in RUN_RESULTS["results"].items():
    test_rows.append({
        "key": key,
        "model": value["display_name"],
        "family": value["family"],
        "accuracy": value["test"]["accuracy"],
        "macro_f1": value["test"]["macro_f1"],
        "majority_accuracy": value["majority_test"]["accuracy"],
        "parameters_m": value["parameter_count"] / 1e6,
        "feature_seconds": (
            value["timing_seconds"]["train_features"]
            + value["timing_seconds"]["validation_features"]
            + value["timing_seconds"]["test_features"]
        ),
        "head_training_seconds": value["timing_seconds"]["head_training"],
        "input_size": "×".join(map(str, value["preprocessing"]["input_size"][-2:])),
        "selected_on_validation": key == SELECTED_MODEL_KEY,
    })
test_table = pd.DataFrame(test_rows).sort_values(["macro_f1", "accuracy"], ascending=False)
display(test_table)


## 6. Error analysis and engineering trade-offs

A useful model comparison needs more than one headline metric.

- **Accuracy** answers how often the top predicted class is correct.
- **Macro-F1** gives every species equal weight.
- **Per-class recall** shows whether aggregate scores hide a weak species.
- **Parameter count** approximates model scale, not runtime by itself.
- **Feature-extraction time** is measured in this session and is hardware-dependent.
- **Input resolution** helps explain why some models cost more per image.

The chart below therefore separates predictive quality from execution cost rather than collapsing them into one score.


In [ ]:
# @title 6.1 Plot quality versus feature-extraction time
fig = plt.figure(figsize=(9, 5))
for row in test_rows:
    plt.scatter(row["feature_seconds"], row["macro_f1"], s=max(40, min(300, row["parameters_m"])))
    plt.annotate(row["key"], (row["feature_seconds"], row["macro_f1"]), xytext=(5, 4), textcoords="offset points")
plt.xlabel("Frozen feature extraction time in this run (seconds)")
plt.ylabel("Independent test macro-F1")
plt.title("Transfer quality versus measured feature-extraction cost")
plt.grid(alpha=0.2)
plt.tight_layout()
tradeoff_plot = OUTPUT_ROOT / "image_classification_quality_vs_runtime.png"
fig.savefig(tradeoff_plot, dpi=140)
plt.show()

recall_rows = []
for key, value in RUN_RESULTS["results"].items():
    for label, recall in value["test"]["per_class_recall"].items():
        recall_rows.append({"model": key, "class": label, "recall": recall})
recall_table = pd.DataFrame(recall_rows).pivot(index="class", columns="model", values="recall")
display(recall_table)


## 7. Inspect one model's mistakes

The table below lists the selected model's independent-test errors. A good follow-up is to open the corresponding images and ask whether failures are associated with pose, background, scale, occlusion, or visually similar species.

This is descriptive error analysis. The notebook does not change the model after looking at test errors.


In [ ]:
# @title 7.1 Selected-model test mistakes
selected_predictions = pd.DataFrame(RUN_RESULTS["results"][SELECTED_MODEL_KEY]["predictions"]["test"])
mistakes = selected_predictions[selected_predictions["truth"] != selected_predictions["prediction"]].copy()
print({"selected_model": MODELS[SELECTED_MODEL_KEY]["display_name"], "test_errors": len(mistakes), "test_rows": len(selected_predictions)})
display(mistakes.head(20))


## 8. Export results and provenance

Each model has already written:

- `head.safetensors` — the trained linear head;
- `manifest.json` — base checkpoint identity, class vocabulary, preprocessing, adaptation recipe, and adapter digest.

These are **workshop transfer-learning adapters**, not replacements for the model-specific production artifact contracts in the six primary DIMER repositories.

The notebook now exports the common comparison tables and provenance.


In [ ]:
# @title 8.1 Machine-readable workshop exports
validation_table.to_csv(OUTPUT_ROOT / "multimodel_image_classification_validation.csv", index=False)
test_table.to_csv(OUTPUT_ROOT / "multimodel_image_classification_test.csv", index=False)
recall_table.to_csv(OUTPUT_ROOT / "multimodel_image_classification_per_class_recall.csv")
selected_predictions.to_csv(OUTPUT_ROOT / "selected_model_test_predictions.csv", index=False)
(OUTPUT_ROOT / "multimodel_image_classification_selection.json").write_text(
    json.dumps(selection_record, indent=2), encoding="utf-8"
)

provenance = {
    "notebook": {
        "specification": "2.1",
        "profile": "E2E",
        "mode": "WORKSHOP",
        "standalone": True,
        "host_repository": "kurtvalcorza/vit-classification-pipeline",
        "generator": "tools/build_multimodel_image_classification_workshop.py",
        "clean_runtime_evidence": "pending",
    },
    "dataset": {
        "kind": sample_kind,
        "manifest_sha256": dataset_sha256,
        "rows": int(len(dataset_manifest)),
        "classes": class_names,
        "split_counts": dataset_manifest["split"].value_counts().to_dict(),
        "default_sample_license": CORPUS_LICENSE if not USE_BYOD else None,
    },
    "adaptation": {
        "method": "linear head on frozen L2-normalized backbone features",
        "head_epochs_budget": HEAD_EPOCHS,
        "head_lr": HEAD_LR,
        "selection": selection_record,
    },
    "runtime": {**RUN_RESULTS["runtime"], "wall_seconds": WALL_SECONDS, "gpu": GPU_INFO},
    "models": {
        key: {
            "declared": MODELS[key],
            "resolved_revision": value["resolved_revision"],
            "parameter_count": value["parameter_count"],
            "feature_dim": value["feature_dim"],
            "preprocessing": value["preprocessing"],
            "selected_epoch": value["selected_epoch"],
            "artifact": value["artifact"],
            "timing_seconds": value["timing_seconds"],
        }
        for key, value in RUN_RESULTS["results"].items()
    },
}
(OUTPUT_ROOT / "multimodel_image_classification_provenance.json").write_text(
    json.dumps(provenance, indent=2), encoding="utf-8"
)
print(sorted(path.name for path in OUTPUT_ROOT.iterdir()))


## 9. Interpretation and limits

This workshop compares **transfer representations under one frozen-backbone linear-probe policy**. It does not establish an overall best image-classification architecture.

Important limits:

- the sample has only 180 photographs and six bird species;
- some species may have appeared in or near the models' pretraining data; overlap cannot be ruled out;
- one deterministic split is not an uncertainty estimate;
- the models differ in pretraining dataset, pretraining objective, input resolution, parameter count, and architecture;
- the standardized probe deliberately prevents each architecture from using its own optimal fine-tuning recipe;
- feature-extraction time is specific to this session's GPU and cache state;
- the workshop head scores are ordinary softmax outputs and are **not calibrated probabilities**; and
- a deployment decision needs repeated splits or external datasets, calibration where probabilities matter, latency/memory tests on target hardware, and domain-specific error analysis.

### Suggested exercises

1. Repeat the experiment with your own pre-split dataset through `BYOD_ZIP_PATH`.
2. Compare the validation and test ordering. Does the selected model remain near the top?
3. Increase or decrease the number of training images per class.
4. Add DINOv2 as a **representation-only** optional condition and label it separately from the pretrained classifiers.
5. In a model-specific DIMER notebook, unfreeze later backbone blocks and compare the gain against the standardized linear probe here.

### References

- DIMER Notebook Specification 2.1: `ml-worker/integrations/dimer/fleet-specs/NOTEBOOK_SPEC.md`
- DIMER model fleet: `ml-worker/integrations/dimer/fleet-inventory/MODEL_MATRIX.md`
- Default sample provenance is carried from `vit-classification-pipeline/src/vit_classification_pipeline/samples.py`.


In [ ]:
# @title Run-all completion summary
summary = {
    "notebook_spec": "2.1",
    "profile": "E2E",
    "mode": "WORKSHOP",
    "models_completed": list(RUN_RESULTS["results"]),
    "selected_on_validation": selection_record["selected_model"],
    "adapter_directories": sorted(path.name for path in ARTIFACT_ROOT.iterdir()),
    "outputs": sorted(path.name for path in OUTPUT_ROOT.iterdir()),
    "clean_runtime_evidence": "pending until exact hosted-runtime execution is recorded",
}
display(pd.Series(summary, name="value").to_frame())
print(
    "Run-all complete: data validation, frozen feature extraction, linear-head adaptation, "
    "validation selection, fresh adapter reload, independent test evaluation, and export succeeded."
)
